In [1]:
import pandas as pd
import torchvision.transforms as T
from PIL import Image
import torch
import torch.nn as nn
import easyocr
import cv2

In [2]:
NUM_EPOCHS = 30
TRAIN_DATA = "traindata.csv"
TEST_DATA = "testdata.csv"

In [3]:
train_df = pd.read_csv(TRAIN_DATA)
test_df  = pd.read_csv(TEST_DATA)
full_df = pd.concat([train_df, test_df], ignore_index=True)

In [4]:
chars = sorted(set("".join(full_df["GroundTruth"].astype(str))))
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
idx_to_char = {i + 1: c for i, c in enumerate(chars)}
BLANK_IDX   = 0
NUM_CLASSES = len(chars) + 1
print("Vocab size (incl. blank):", NUM_CLASSES)

Vocab size (incl. blank): 37


In [5]:
val_transform = T.Compose([
    T.Resize((32, 256)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

In [6]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            # Layer 1: 32x256 -> 16x128
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Layer 2: 16x128 -> 8x64
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Layer 3: 8x64 -> 8x64
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Layer 4: 8x64 -> 8x64
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Squash height from 8 down to 1 while preserving 64 horizontal time steps
            nn.MaxPool2d(kernel_size=(8, 1)) 
        )
        self.rnn = nn.LSTM(
            input_size=256, hidden_size=256,
            num_layers=2, bidirectional=True, batch_first=False,dropout=0.5
        )
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)          # Output shape: (B, 256, 1, 64)
        x = x.squeeze(2)         # Output shape: (B, 256, 64)
        x = x.permute(2, 0, 1)   # Output shape: (64, B, 256) -> Time-first for LSTM
        x, _ = self.rnn(x)       # Output shape: (64, B, 512)
        x = self.fc(x)           # Output shape: (64, B, num_classes)
        return x

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
model = CRNN(num_classes=NUM_CLASSES)
model_path = "crnn_trained.pth"
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

print(f"Successfully loaded weights from '{model_path}' onto {device}!")

Using CPU. Note: This module is much faster with a GPU.


Successfully loaded weights from 'crnn_trained.pth' onto cpu!


In [8]:
def decode_single(pred_sequence):
    """Greedy CTC decode for a 1-D array of class indices."""
    output, prev = [], -1
    for p in pred_sequence:
        if p != prev and p != BLANK_IDX:
            output.append(idx_to_char[p])
        prev = p
    return "".join(output)

def decode_batch(logits):
    """
    logits: (T, B, num_classes)
    returns: list of decoded strings, one per sample
    """
    pred = logits.argmax(2).cpu().numpy()   # (T, B)
    return [decode_single(pred[:, b]) for b in range(pred.shape[1])]

def predict_image(path):
    image = Image.open(path).convert("L")
    image = val_transform(image).unsqueeze(0).to(device)  # (1, 1, 32, 256)
    model.eval()
    with torch.no_grad():
        logits = model(image)       # (T, 1, num_classes)
    return decode_batch(logits)[0]  # single string

In [9]:
def pipeline_ocr(image_path):
    print(f"Processing: {image_path}")
    
    # Load image via OpenCV
    orig_image = cv2.imread(image_path)
    if orig_image is None:
        print("Error: Image not found!")
        return []
        
    # Stage A: Run CRAFT Detector Only
    # reader.detect() returns bounding boxes in format: [ [x_min, x_max, y_min, y_max], ... ]
    horizontal_boxes, free_boxes = reader.detect(image_path)
    detected_boxes = horizontal_boxes[0] # List of detected text coordinates
    
    final_transcriptions = []
    
    # Stage B: Crop, Transform, and Recognize
    for i, box in enumerate(detected_boxes):
        x_min, x_max, y_min, y_max = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        
        # Crop the word patch from the image matrix safely
        crop = orig_image[y_min:y_max, x_min:x_max]
        if crop.size == 0:
            continue
            
        # Process patch for your CRNN (BGR -> Grayscale -> PIL Image)
        crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        crop_pil = Image.fromarray(crop_gray)
        
        # Transform image to match network inputs: (1, 1, 32, 256)
        img_tensor = val_transform(crop_pil).unsqueeze(0).to(device)
        
        # Forward pass through your fine-tuned model
        with torch.no_grad():
            logits = model(img_tensor)
            raw_pred = decode_batch(logits)[0] # Extract decoded string
            
        # Optional: Apply your lexicon correction script here if you have a local dictionary!
        # raw_pred = edit_distance_correction(raw_pred, current_lexicon)
            
        print(f" Detected Box {i} [{x_min},{y_min}]: '{raw_pred}'")
        final_transcriptions.append({
            "box": [x_min, y_min, x_max, y_max],
            "text": raw_pred
        })
        
    return final_transcriptions

In [10]:
pipeline_ocr("/Users/pashantraj/Desktop/Repos/Computer_vision/CRNN/train/5186_8.png")

Processing: /Users/pashantraj/Desktop/Repos/Computer_vision/CRNN/train/5186_8.png
 Detected Box 0 [13,11]: 'WRIGHT'


[{'box': [13, 11, 217, 53], 'text': 'WRIGHT'}]

In [11]:
print(model)

CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): R